# Installing Dependencies


In [1]:
# Required libraries for this notebook. LightGBM is the main model; sklearn models are used as robust ensemble members.
%pip -q install lightgbm



[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Data Load


In [2]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

try:
    from lightgbm import LGBMClassifier
    HAS_LIGHTGBM = True
except Exception as e:
    HAS_LIGHTGBM = False
    print('LightGBM import failed; using sklearn fallback models only:', repr(e))

DATA_DIR = Path('.')

def read_csv_first(candidates):
    for name in candidates:
        path = DATA_DIR / name
        if path.exists():
            print(f'Loaded {name}')
            return pd.read_csv(path)
    raise FileNotFoundError(f'None of these files were found: {candidates}')

TRAIN_DATA = read_csv_first(['train-sensor.csv', 'train-data.csv'])
TRAIN_LABEL = read_csv_first(['train-label.csv'])
TEST_DATA = read_csv_first(['test-sensor.csv', 'test-data.csv'])
TEST_LABEL = read_csv_first(['test-label.csv']) if (DATA_DIR / 'test-label.csv').exists() else None

for df in [TRAIN_DATA, TRAIN_LABEL, TEST_DATA, TEST_LABEL]:
    if df is not None and 'id' in df.columns:
        df.set_index('id', inplace=True, drop=True)

TARGET_COL = 'stress' if 'stress' in TRAIN_LABEL.columns else TRAIN_LABEL.columns[0]
CLASSES = np.sort(TRAIN_LABEL[TARGET_COL].dropna().unique()).astype(int)
print('Target:', TARGET_COL, '| classes:', CLASSES.tolist())
print('TRAIN_DATA', TRAIN_DATA.shape, 'TRAIN_LABEL', TRAIN_LABEL.shape, 'TEST_DATA', TEST_DATA.shape)
print(TRAIN_LABEL[TARGET_COL].value_counts(normalize=True).sort_index())


Loaded train-sensor.csv
Loaded train-label.csv
Loaded test-sensor.csv
Loaded test-label.csv
Target: stress | classes: [0, 1, 2]
TRAIN_DATA (4694400, 8) TRAIN_LABEL (815, 3) TEST_DATA (5921280, 8)
stress
0.0    0.198773
1.0    0.080982
2.0    0.720245
Name: proportion, dtype: float64


In [3]:
# Feature engineering and model training.
# For the E4 stress task, each label is represented by sensor statistics from
# windows immediately before that label timestamp. Validation is grouped by pid.

EXCLUDE_COLS = {'id', 'pid', 'timestamp', TARGET_COL}
IS_SENSOR_TASK = (
    {'pid', 'timestamp'}.issubset(TRAIN_DATA.columns)
    and {'pid', 'timestamp', TARGET_COL}.issubset(TRAIN_LABEL.columns)
)
WINDOWS_MS = [60_000, 180_000, 300_000]

def add_sensor_derived_columns(df):
    df = df.copy()
    for c in df.columns:
        if c not in EXCLUDE_COLS:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    if {'accel_x', 'accel_y', 'accel_z'}.issubset(df.columns):
        ax, ay, az = df['accel_x'], df['accel_y'], df['accel_z']
        df['accel_mag'] = np.sqrt(ax * ax + ay * ay + az * az)
        df['accel_xy'] = np.sqrt(ax * ax + ay * ay)
    return df

def summarize_array(values, prefix):
    values = pd.Series(values).dropna().astype(float)
    stats = ['mean', 'std', 'min', 'max', 'median', 'q25', 'q75', 'iqr', 'range', 'first', 'last', 'delta', 'absdiff_mean', 'absdiff_sum']
    if len(values) == 0:
        return {f'{prefix}_{s}': np.nan for s in stats}
    q25, q75 = values.quantile([0.25, 0.75])
    diffs = values.diff().dropna().abs()
    return {
        f'{prefix}_mean': values.mean(),
        f'{prefix}_std': values.std(ddof=0),
        f'{prefix}_min': values.min(),
        f'{prefix}_max': values.max(),
        f'{prefix}_median': values.median(),
        f'{prefix}_q25': q25,
        f'{prefix}_q75': q75,
        f'{prefix}_iqr': q75 - q25,
        f'{prefix}_range': values.max() - values.min(),
        f'{prefix}_first': values.iloc[0],
        f'{prefix}_last': values.iloc[-1],
        f'{prefix}_delta': values.iloc[-1] - values.iloc[0],
        f'{prefix}_absdiff_mean': diffs.mean() if len(diffs) else 0.0,
        f'{prefix}_absdiff_sum': diffs.sum() if len(diffs) else 0.0,
    }

def extract_window_features(sensor_df, label_df, windows_ms=WINDOWS_MS):
    sensor = add_sensor_derived_columns(sensor_df)
    labels = label_df.copy()
    sensor['timestamp'] = pd.to_numeric(sensor['timestamp'], errors='coerce')
    labels['timestamp'] = pd.to_numeric(labels['timestamp'], errors='coerce')
    sensor_cols = [c for c in sensor.columns if c not in EXCLUDE_COLS and pd.api.types.is_numeric_dtype(sensor[c])]
    by_pid = {pid: g.sort_values('timestamp') for pid, g in sensor.groupby('pid')}

    rows = []
    for n, (label_id, lab) in enumerate(labels.iterrows(), start=1):
        pid, ts = lab['pid'], lab['timestamp']
        subj = by_pid.get(pid)
        row = {'id': label_id, 'pid': pid, 'timestamp': ts}
        if subj is None or pd.isna(ts):
            rows.append(row)
            continue
        base_mean = subj[sensor_cols].mean()
        base_std = subj[sensor_cols].std(ddof=0).replace(0, np.nan)
        for w in windows_ms:
            win = subj[(subj['timestamp'] <= ts) & (subj['timestamp'] > ts - w)]
            suffix = f'w{w // 1000}s'
            row[f'{suffix}_count'] = len(win)
            row[f'{suffix}_span_ms'] = (win['timestamp'].max() - win['timestamp'].min()) if len(win) else 0
            for col in sensor_cols:
                row.update(summarize_array(win[col].values, f'{col}_{suffix}'))
                if len(win):
                    z = (win[col] - base_mean[col]) / base_std[col]
                    row.update(summarize_array(z.values, f'{col}_z_{suffix}'))
        rows.append(row)
        if n % 200 == 0:
            print(f'  extracted {n}/{len(labels)} label windows')
    return pd.DataFrame(rows).set_index('id')

def make_tabular_features(train_df, test_df):
    train, test = train_df.copy(), test_df.copy()
    if TARGET_COL in train.columns:
        train = train.drop(columns=[TARGET_COL])
    full = pd.concat([train, test], axis=0)
    cat_cols = full.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    num_cols = [c for c in full.columns if c not in cat_cols]
    if cat_cols:
        enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        full[cat_cols] = enc.fit_transform(full[cat_cols].astype(str))
    full[num_cols] = full[num_cols].apply(pd.to_numeric, errors='coerce')
    return full.iloc[:len(train)].copy(), full.iloc[len(train):].copy(), None

if IS_SENSOR_TASK:
    print('Detected sensor stress schema. Extracting timestamp-window features...')
    if TEST_LABEL is None:
        raise ValueError('test-label.csv is required because its id/pid/timestamp rows define the submission examples.')
    X_train = extract_window_features(TRAIN_DATA, TRAIN_LABEL)
    X_test = extract_window_features(TEST_DATA, TEST_LABEL)
    groups = TRAIN_LABEL['pid'].astype(str).values
else:
    print('Detected direct tabular schema. Using encoded row-level features.')
    X_train, X_test, groups = make_tabular_features(TRAIN_DATA, TEST_DATA)
    if 'pid' in TRAIN_DATA.columns:
        groups = TRAIN_DATA['pid'].astype(str).values

y = TRAIN_LABEL[TARGET_COL].astype(int).values
feature_cols = sorted(set(X_train.columns) | set(X_test.columns))
X_train = X_train.reindex(columns=feature_cols)
X_test = X_test.reindex(columns=feature_cols)
X_train_model = X_train.drop(columns=['pid'], errors='ignore')
X_test_model = X_test.drop(columns=['pid'], errors='ignore')

imputer = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train_model), index=X_train_model.index, columns=X_train_model.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test_model), index=X_test_model.index, columns=X_test_model.columns)
selector = VarianceThreshold(threshold=0.0)
X_arr = selector.fit_transform(X_train_imp)
X_te_arr = selector.transform(X_test_imp)
print('Final feature matrix:', X_arr.shape, X_te_arr.shape)

if groups is not None and len(np.unique(groups)) >= 3:
    n_splits = min(5, len(np.unique(groups)))
    splits = list(GroupKFold(n_splits=n_splits).split(X_arr, y, groups=groups))
    print(f'CV: GroupKFold by pid, n_splits={n_splits}')
else:
    class_counts = np.bincount(y)
    n_splits = max(3, min(5, class_counts[class_counts > 0].min()))
    splits = list(StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED).split(X_arr, y))
    print(f'CV: StratifiedKFold, n_splits={n_splits}')

models = []
if HAS_LIGHTGBM:
    models.append(('lgbm', LGBMClassifier(
        objective='multiclass' if len(CLASSES) > 2 else 'binary',
        n_estimators=700, learning_rate=0.025, num_leaves=15, max_depth=4,
        min_child_samples=35, subsample=0.85, colsample_bytree=0.8,
        reg_alpha=1.5, reg_lambda=8.0, class_weight='balanced',
        random_state=RANDOM_SEED, n_jobs=-1, verbose=-1,
    )))
models.extend([
    ('extra_trees', ExtraTreesClassifier(
        n_estimators=700, max_depth=10, min_samples_leaf=8, max_features=0.65,
        class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1,
    )),
    ('hist_gb', HistGradientBoostingClassifier(
        learning_rate=0.045, max_iter=450, max_leaf_nodes=15, max_depth=5,
        l2_regularization=1.0, random_state=RANDOM_SEED,
    )),
])

n_classes = len(CLASSES)
oof = {name: np.zeros((len(y), n_classes)) for name, _ in models}
test_pred = {name: np.zeros((len(X_te_arr), n_classes)) for name, _ in models}

for fold, (tr_idx, val_idx) in enumerate(splits, start=1):
    X_tr, X_val = X_arr[tr_idx], X_arr[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    print(f'Fold {fold}/{len(splits)} | train={len(tr_idx)} val={len(val_idx)}')
    for name, proto in models:
        model = clone(proto)
        fit_kwargs = {'sample_weight': compute_sample_weight('balanced', y_tr)} if name == 'hist_gb' else {}
        model.fit(X_tr, y_tr, **fit_kwargs)
        val_proba = model.predict_proba(X_val)
        te_proba = model.predict_proba(X_te_arr)
        aligned_val = np.zeros((len(val_idx), n_classes))
        aligned_te = np.zeros((len(X_te_arr), n_classes))
        for j, cls in enumerate(model.classes_.astype(int)):
            pos = np.where(CLASSES == cls)[0][0]
            aligned_val[:, pos] = val_proba[:, j]
            aligned_te[:, pos] = te_proba[:, j]
        oof[name][val_idx] = aligned_val
        test_pred[name] += aligned_te / len(splits)
        print(f'  {name:12s} fold BA={balanced_accuracy_score(y_val, CLASSES[aligned_val.argmax(axis=1)]):.4f}')

print('\nOOF balanced accuracy by model')
for name in oof:
    print(f'{name:12s}: {balanced_accuracy_score(y, CLASSES[oof[name].argmax(axis=1)]):.5f}')

names = list(oof)
best_score, best_weights = -1, None
if len(names) == 1:
    best_weights = np.array([1.0])
else:
    grid = np.linspace(0, 1, 11)
    for a in grid:
        for b in grid:
            weights = np.array([a, b, 1 - a - b])[:len(names)]
            if len(names) == 2:
                weights = np.array([a, 1 - a])
            if np.any(weights < -1e-9) or abs(weights.sum() - 1) > 1e-9:
                continue
            blend = sum(weights[i] * oof[names[i]] for i in range(len(names)))
            score = balanced_accuracy_score(y, CLASSES[blend.argmax(axis=1)])
            if score > best_score:
                best_score, best_weights = score, weights

blend_oof = sum(best_weights[i] * oof[names[i]] for i in range(len(names)))
blend_test = sum(best_weights[i] * test_pred[names[i]] for i in range(len(names)))
OOF_BA = balanced_accuracy_score(y, CLASSES[blend_oof.argmax(axis=1)])
TEST_PREDICTIONS = CLASSES[blend_test.argmax(axis=1)].astype(int)

print('\nSelected ensemble weights')
for n, w in zip(names, best_weights):
    print(f'{n:12s}: {w:.2f}')
print(f'Ensemble OOF balanced accuracy: {OOF_BA:.5f}')
print('Prediction distribution:', dict(pd.Series(TEST_PREDICTIONS).value_counts().sort_index()))


Detected sensor stress schema. Extracting timestamp-window features...
  extracted 200/815 label windows
  extracted 400/815 label windows
  extracted 600/815 label windows
  extracted 800/815 label windows
  extracted 200/1028 label windows
  extracted 400/1028 label windows
  extracted 600/1028 label windows
  extracted 800/1028 label windows
  extracted 1000/1028 label windows
Final feature matrix: (815, 679) (1028, 679)
CV: GroupKFold by pid, n_splits=5
Fold 1/5 | train=663 val=152
  lgbm         fold BA=0.5000
  extra_trees  fold BA=0.5000
  hist_gb      fold BA=0.5000
Fold 2/5 | train=671 val=144
  lgbm         fold BA=0.3071
  extra_trees  fold BA=0.3333
  hist_gb      fold BA=0.3095
Fold 3/5 | train=678 val=137
  lgbm         fold BA=0.3993
  extra_trees  fold BA=0.5000
  hist_gb      fold BA=0.4328
Fold 4/5 | train=616 val=199
  lgbm         fold BA=0.5402
  extra_trees  fold BA=0.5415
  hist_gb      fold BA=0.5011
Fold 5/5 | train=632 val=183
  lgbm         fold BA=0.3244
  e

In [4]:
# Kaggle submission file. Stress task requires columns: id,stress.
submission = pd.DataFrame({
    'id': X_test.index,
    TARGET_COL: TEST_PREDICTIONS.astype(int),
})
submission.to_csv('submissionc.csv', index=False)
print('Saved submission.csv')
print(submission.head())
print(submission[TARGET_COL].value_counts().sort_index())


Saved submission.csv
     id  stress
0  1227       2
1  1228       2
2  1229       2
3  1230       2
4  1231       2
stress
0    269
1     48
2    711
Name: count, dtype: int64
